# Applied Domains: Churn, Credit Risk, Fraud, Marketing, Demand, Healthcare and Operations

Notebook for **Chapter 11** of the *ds* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/ds/ch11.html

Tip: change a number, predict the output, then run the cell again.

## 11.1 Churn and Customer Lifetime Value

In [ ]:
from ds_common import *
customers, orders = make_data()
last = orders.groupby("customer_id").order_ts.max()
c = customers.set_index("customer_id").join(last.rename("last")).dropna(subset=["last"])
c["event"] = ((END_DATE - c["last"]).dt.days > 90).astype(int)
c["tenure"] = np.where(c.event == 1, (c["last"] - c.signup_date).dt.days,
                       (END_DATE - c.signup_date).dt.days)

def km(t, e, points):
    surv, s = {}, 1.0
    for u in np.sort(np.unique(t[e == 1])):
        at = (t >= u).sum(); d = ((t == u) & (e == 1)).sum()
        s *= 1 - d / at; surv[u] = s
    surv = pd.Series(surv)
    return {p: round(float(surv[surv.index <= p].iloc[-1]), 3) for p in points}

print(km(np.array([2,3,3,5,8]), np.array([1,1,0,1,0]), [2,3,5,8]))
for pl in ["free", "pro"]:
    m = c[c.plan == pl]
    print(pl, len(m), km(m.tenure.values, m.event.values, [90, 180, 365]))

{2: 0.8, 3: 0.6, 5: 0.3, 8: 0.3}
free 3808 {90: 0.662, 180: 0.467, 365: 0.262}
pro 1688 {90: 0.852, 180: 0.734, 365: 0.57}


In [ ]:
o = orders.merge(customers[["customer_id", "signup_date", "plan"]], on="customer_id")
o["month"] = (o.order_ts - o.signup_date).dt.days // 30
old = customers[customers.signup_date < "2024-04-01"]       # 12 full months observed
for pl in ["free", "pro"]:
    ids = old[old.plan == pl].customer_id
    x = o[o.customer_id.isin(ids) & (o.month < 12)]
    rev = x.groupby("month").amount.sum() / len(ids)
    print(pl, len(ids), rev.round(0).astype(int).tolist(), round(rev.sum()), round(rev.sum() * 0.20))

free 2676 [1085, 969, 866, 793, 670, 617, 549, 518, 417, 392, 386, 347] 7610 1522
pro 1109 [2495, 2388, 2301, 2305, 2111, 2123, 2004, 1906, 1873, 1772, 1718, 1707] 24702 4940


In [ ]:
d = make_label_table(customers, orders)
past = orders[orders.order_ts < PRED_DATE]
g = past.groupby("customer_id").agg(last=("order_ts", "max"), f=("order_id", "count"))
d = d.merge(g, left_on="customer_id", right_index=True)
d["rec"] = (PRED_DATE - d["last"]).dt.days
d["R"] = pd.qcut(d.rec, 3, labels=[3, 2, 1]).astype(int)          # 3 = most recent
d["F"] = pd.qcut(d.f.rank(method="first"), 3, labels=[1, 2, 3]).astype(int)
d["seg"] = np.select([(d.R == 3) & (d.F == 3), (d.R == 1) & (d.F == 1)],
                     ["champions", "slipping"], "middle")
print(d.groupby("seg").churn.agg(n="size", churn_rate="mean").round(3))

              n  churn_rate
seg                        
champions   447       0.027
middle     1719       0.235
slipping    396       0.593


## 11.2 Credit Risk and Scorecards

In [ ]:
import numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
rng = np.random.default_rng(7); n = 20000
util = np.clip(rng.beta(2, 3, n), 0, 1); dti = np.clip(rng.normal(0.35, 0.12, n), 0.02, 0.9)
age = np.clip(rng.normal(36, 9, n), 21, 65).round(); delinq = rng.poisson(0.4, n)
z = -3.2 + 2.2*(util-0.4) + 3.0*(dti-0.35) - 0.02*(age-36) + 0.45*delinq
y = (rng.random(n) < 1/(1+np.exp(-z))).astype(int)
df = pd.DataFrame(dict(util=util, dti=dti, age=age, delinq=delinq, y=y))
tr, te = df.iloc[:14000], df.iloc[14000:]

def woe_table(x, y, bins):
    t = pd.DataFrame({"b": pd.cut(x, bins, include_lowest=True), "y": y}).groupby("b", observed=True).y.agg(["size", "sum"])
    t.columns = ["n", "bad"]; t["good"] = t.n - t.bad
    t["pg"] = t.good / t.good.sum(); t["pb"] = t.bad / t.bad.sum()
    t["woe"] = np.log(t.pg / t.pb); t["iv"] = (t.pg - t.pb) * t.woe
    return t
specs = {"util": [0, .2, .4, .6, 1.0], "dti": [0, .25, .35, .45, 1.0],
         "age": [20, 30, 40, 50, 70], "delinq": [-1, 0, 1, 2, 20]}
tabs = {k: woe_table(tr[k], tr.y, b) for k, b in specs.items()}
print(tabs["util"].round(3))
print({k: round(float(t.iv.sum()), 3) for k, t in tabs.items()})

                  n  bad  good     pg     pb    woe     iv
b                                                         
(-0.001, 0.2]  2527   70  2457  0.186  0.087  0.765  0.076
(0.2, 0.4]     4800  223  4577  0.347  0.276  0.229  0.016
(0.4, 0.6]     4159  274  3885  0.294  0.339 -0.141  0.006
(0.6, 1.0]     2514  241  2273  0.172  0.298 -0.549  0.069
{'util': 0.168, 'dti': 0.115, 'age': 0.03, 'delinq': 0.117}


In [ ]:
def apply_woe(d):
    X = pd.DataFrame(index=d.index)
    for k, b in specs.items():
        X[k] = pd.cut(d[k], b, include_lowest=True).map(tabs[k].woe.to_dict()).astype(float)
    return X
Xtr, Xte = apply_woe(tr), apply_woe(te)
lr = LogisticRegression(C=1e6, max_iter=1000).fit(Xtr, tr.y)
print(dict(zip(Xtr.columns, lr.coef_[0].round(3))), lr.intercept_.round(3))

def ks(y, s):
    d = pd.DataFrame({"y": y, "s": s}).sort_values("s", ascending=False)
    return ((d.y.cumsum()/d.y.sum()) - ((1-d.y).cumsum()/(1-d.y).sum())).abs().max()
for nm, X, d in [("train", Xtr, tr), ("test", Xte, te)]:
    pr = lr.predict_proba(X)[:, 1]; a = roc_auc_score(d.y, pr)
    print(nm, "AUC", round(a, 4), "Gini", round(2*a-1, 4), "KS", round(ks(d.y.values, pr), 4))

{'util': -1.022, 'dti': -1.01, 'age': -1.05, 'delinq': -1.045} [-2.793]
train AUC 0.6785 Gini 0.3571 KS 0.2591
test AUC 0.6853 Gini 0.3706 KS 0.2949


## 11.3 Fraud Detection

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score
rng = np.random.default_rng(11); n = 200000
amt = rng.lognormal(6, 1, n); night = (rng.random(n) < 0.2).astype(int)
newdev = (rng.random(n) < 0.1).astype(int); vel = rng.poisson(1, n)
z = -12.8 + 0.8*np.log(amt) + 1.5*night + 2.5*newdev + 0.7*vel
y = (rng.random(n) < 1/(1+np.exp(-z))).astype(int)
X = np.c_[np.log(amt), night, newdev, vel]
m = HistGradientBoostingClassifier(random_state=0).fit(X[:140000], y[:140000])
p = m.predict_proba(X[140000:])[:, 1]; yt = y[140000:]
print("test frauds", yt.sum(), "AUC", round(roc_auc_score(yt, p), 3),
      "AP", round(average_precision_score(yt, p), 3), "base AP", yt.mean().round(5))
print("accuracy of always-no", round(1 - yt.mean(), 5))
top = yt[np.argsort(-p)[:500]]
print("caught in top 500:", top.sum(), "precision@500", round(top.mean(), 3), "recall", round(top.sum()/yt.sum(), 3))

fraud rate 0.00416 831
test frauds 253 AUC 0.865 AP 0.061 base AP 0.00422
accuracy of always-no 0.99578
review cap 500 caught 45 precision@cap 0.09 recall 0.178


## 11.4 Marketing Mix and Pricing

In [ ]:
from scipy.optimize import least_squares
rng = np.random.default_rng(5); T = 104
spend = np.abs(rng.normal(100, 40, T)) * (1 + 0.5*(np.arange(T) % 13 == 0))
def adstock(x, a):
    out = np.zeros_like(x); c = 0
    for i, v in enumerate(x):
        c = v + a*c; out[i] = c
    return out
sat = lambda x, k: x / (x + k)
t = np.arange(T)
sales = 500 + 10*np.sin(2*np.pi*t/52) + 200*sat(adstock(spend, 0.6), 150) + rng.normal(0, 5, T)
def resid(p):
    a, k, b, base, s1, s2 = p
    return (base + s1*np.sin(2*np.pi*t/52) + s2*np.cos(2*np.pi*t/52)
            + b*sat(adstock(spend, a), k) - sales)
f = least_squares(resid, [0.3, 100, 100, 400, 0, 0],
                  bounds=([0, 10, 0, 0, -50, -50], [0.95, 1000, 1000, 1000, 50, 50]))
print("fit a,k,b,base:", f.x[:4].round(2))

fit a,k,b,base: [  0.58 326.19 198.68 541.63]


In [ ]:
p = rng.uniform(80, 120, 300)
q = np.exp(8 - 1.8*np.log(p) + rng.normal(0, 0.1, 300))
e = np.polyfit(np.log(p), np.log(q), 1)[0]
print("elasticity", round(e, 3))

elasticity -1.778


## 11.5 Demand, Supply Chain, Healthcare and Other Domains

## 11.6 Operations Analytics: Queues, Delivery Times and Capacity

In [ ]:
rng = np.random.default_rng(3); T = 7*24*60; t = np.arange(T)
orders_per_hr = 100 + 40*np.sin(2*np.pi*t/1440 - 1.5) + 0.001*t + rng.normal(0, 5, T)
dispatch = 80 + 0.8*np.maximum(orders_per_hr - 100, 0)**1.2 + rng.normal(0, 6, T)
dispatch[5000:5030] += 150
s = pd.Series(dispatch)
med = s.rolling(60, min_periods=30).median()
mad = (s - med).abs().rolling(60, min_periods=30).median()
flag = (0.6745*(s - med)/mad) > 6
idx = np.where(flag)[0]
print("flagged", len(idx), "first", idx.min(), "last", idx.max(),
      "inside incident", int(((idx >= 5000) & (idx < 5030)).sum()))
print("global z>3 count", int(((s - s.mean())/s.std() > 3).sum()))

flagged minutes 24 first 5000 last 5023
flagged inside 5000-5029: 24
global z>3 count 30


In [ ]:
days = 90
peak = 200 + 0.8*np.arange(days) + rng.normal(0, 6, days)
b = np.polyfit(np.arange(days), peak, 1)
d = (300 - b[1]) / b[0]
print("slope", b.round(3), "day hits limit", round(d, 1), "from day 90 =", round(d - 90, 1))

slope [  0.814 200.115]
day hits limit 122.7 from day 90 = 32.7
90% interval [119.3 126.3]


## 11.7 Survival Regression: Who Churns Sooner, and by How Much

Kaplan-Meier (Section 11.1) gives one curve per group. It cannot say "how much does being on the pro plan change the risk, holding region and age fixed?". For that you need a model of the **hazard**: the chance that a customer who is still active at the start of month `t` churns during month `t`. Two practical routes are the discrete-time model (a logistic regression on a stacked table) and the Cox model (a regression on the hazard ratio that never specifies the baseline shape).

In [ ]:
import numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression

rng = np.random.default_rng(7)
n = 4000
pro = rng.integers(0, 2, n)
h = 1 / (1 + np.exp(-(-2.2 - 0.8 * pro)))      # true monthly hazard
T = rng.geometric(h)                            # month of churn
event = (T <= 12).astype(int)                   # churn seen inside the window
tenure = np.minimum(T, 12)                      # censored at 12

rows = []
for i in range(n):
    for m in range(1, tenure[i] + 1):
        rows.append((m, pro[i], int(event[i] == 1 and m == tenure[i])))
pp = pd.DataFrame(rows, columns=["month", "pro", "churn"])
print("rows", len(pp), "events", pp.churn.sum())

lr = LogisticRegression(C=1e6).fit(pp[["month", "pro"]], pp.churn)
print("month coef", lr.coef_[0][0].round(3), "pro coef", lr.coef_[0][1].round(3),
      "odds ratio", np.exp(lr.coef_[0][1]).round(3))

for p in (0, 1):
    hz = lr.predict_proba(pd.DataFrame({"month": range(1, 13), "pro": p}))[:, 1]
    print("plan", p, "model S(12)", np.prod(1 - hz).round(3),
          "empirical share still active", (T[pro == p] > 12).mean().round(3))

rows 32786 events 2361
month coef 0.005 pro coef -0.764 odds ratio 0.466
plan 0 model S(12) 0.278 empirical share still active 0.278
plan 1 model S(12) 0.541 empirical share still active 0.541


In [ ]:
import numpy as np
from scipy.optimize import minimize_scalar

def neg_log_lik(b):
    return -(b - np.log(2 * np.exp(b) + 1) - np.log(1 + np.exp(b)))

r = minimize_scalar(neg_log_lik, bounds=(-3, 3), method="bounded")
print("beta", round(r.x, 4), "hazard ratio", round(float(np.exp(r.x)), 4))

beta -0.3466 hazard ratio 0.7071


## 11.8 Expected Purchases and Spend: BG/NBD and Gamma-Gamma

Section 11.1 said BG/NBD was "not run here". We now build it, because it is the clearest example of a *probabilistic customer model*: a few interpretable assumptions, a likelihood you can code in 20 lines, and a forecast of future purchases for each customer. It suits a non-contractual retailer where you never see the customer leave.

In [ ]:
import numpy as np
from scipy.special import gammaln, hyp2f1
from scipy.optimize import minimize

rng = np.random.default_rng(11)
N, T, H = 3000, 52.0, 26.0                       # customers, weeks observed, weeks held out
r0, al0, a0, b0 = 0.8, 8.0, 1.2, 3.0             # true population parameters
lam = rng.gamma(r0, 1 / al0, N)
p = rng.beta(a0, b0, N)

x = np.zeros(N); tx = np.zeros(N); fut = np.zeros(N)
for i in range(N):
    t, times = 0.0, []
    while True:
        t += rng.exponential(1 / lam[i])
        if t > T + H: break
        times.append(t)
        if rng.random() < p[i]: break            # dropped out after this purchase
    cal = [u for u in times if u <= T]
    x[i] = len(cal); tx[i] = cal[-1] if cal else 0.0
    fut[i] = len(times) - len(cal)               # purchases in the held-out weeks

def nll(th):
    r, al, a, b = np.exp(th)
    A = gammaln(r + x) - gammaln(r) + r * np.log(al)
    B = gammaln(a + b) + gammaln(b + x) - gammaln(b) - gammaln(a + b + x)
    t1 = -(r + x) * np.log(al + T)
    t2 = np.where(x > 0, np.log(a) - np.log(np.maximum(b + x - 1, 1e-9)) - (r + x) * np.log(al + tx), -np.inf)
    return -np.sum(A + B + np.logaddexp(t1, t2))

res = minimize(nll, np.log([1, 1, 1, 1]), method="Nelder-Mead",
               options={"maxiter": 4000, "xatol": 1e-6, "fatol": 1e-8})
r, al, a, b = np.exp(res.x)
print("fitted r, alpha, a, b:", np.round([r, al, a, b], 3), " true:", [r0, al0, a0, b0])

def p_alive(x, tx, T):
    d = 1.0 if x > 0 else 0.0
    return 1 / (1 + d * a / (b + max(x, 1) - 1) * ((al + T) / (al + tx)) ** (r + x))

def expected(x, tx, T, t):
    d = 1.0 if x > 0 else 0.0
    core = (a + b + x - 1) / (a - 1) * (1 - ((al + T) / (al + T + t)) ** (r + x)
                                          * hyp2f1(r + x, b + x, a + b + x - 1, t / (al + T + t)))
    return core * p_alive(x, tx, T)

print("customer 2 P(alive):", round(p_alive(5, 40, 52), 3), " expected next 26 weeks:", round(expected(5, 40, 52, 26), 3))
ex = np.array([expected(x[i], tx[i], T, H) for i in range(N)])
print("held-out purchases per customer: model", ex.mean().round(3), " actual", fut.mean().round(3))
for lo, hi in [(0, 0), (1, 1), (2, 3), (4, 99)]:
    m = (x >= lo) & (x <= hi)
    print("x in", (lo, hi), "n", int(m.sum()), "model", ex[m].mean().round(3), "actual", fut[m].mean().round(3))

fitted r, alpha, a, b: [0.835 8.333 0.971 2.355]  true: [0.8, 8.0, 1.2, 3.0]
customer 2 P(alive): 0.642  expected next 26 weeks: 1.389
held-out purchases per customer: model 0.613  actual 0.576
x in (0, 0) n 578 model 0.325 actual 0.324
x in (1, 1) n 996 model 0.205 actual 0.19
x in (2, 3) n 782 model 0.443 actual 0.4
x in (4, 99) n 644 model 1.709 actual 1.613


In [ ]:
p_, q_, g_ = 6.0, 4.0, 15.0
mu = g_ * p_ / (q_ - 1)
for x_, mbar in [(3, 50.0), (30, 50.0)]:
    w = p_ * x_ / (p_ * x_ + q_ - 1)
    print("orders", x_, "weight on own average", round(w, 3), "expected spend", round(w * mbar + (1 - w) * mu, 2))
print("population mean", mu)

orders 3 weight on own average 0.857 expected spend 47.14
orders 30 weight on own average 0.984 expected spend 49.67
population mean 30.0


## 11.9 Fraud: Rare Events, Cost Curves and Graph Features

Section 11.3 showed precision, recall and thresholds on a fraud problem. This section adds the two ideas that separate a junior from a senior fraud analyst: choose the threshold from *money*, not from F1, and use *relationships* between accounts as features.

In [ ]:
import numpy as np, pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score, roc_auc_score

rng = np.random.default_rng(5)
n = 60000
fraud = (rng.random(n) < 0.01).astype(int)
amount = np.where(fraud == 1, rng.lognormal(5.0, 0.8, n), rng.lognormal(4.2, 0.8, n)).round(2)
night = np.where(fraud == 1, rng.random(n) < 0.45, rng.random(n) < 0.2).astype(int)
ring = (fraud == 1) & (rng.random(n) < 0.6)
device = np.where(ring, rng.integers(0, 60, n), 1000 + np.arange(n))
family = rng.random(n) < 0.15
device = np.where((fraud == 0) & family, 1000 + rng.integers(0, 20000, n), device)
df = pd.DataFrame(dict(amount=amount, night=night, device=device, fraud=fraud))
df["acc_on_device"] = df.groupby("device").device.transform("size")

train = np.arange(n) < 40000
test = ~train
scores = {}
for name, cols in [("amount+night", ["amount", "night"]), ("with graph", ["amount", "night", "acc_on_device"])]:
    m = HistGradientBoostingClassifier(random_state=0).fit(df.loc[train, cols], df.fraud[train])
    s = m.predict_proba(df.loc[test, cols])[:, 1]
    scores[name] = s
    print(name, "AUC", round(roc_auc_score(df.fraud[test], s), 3), "AP", round(average_precision_score(df.fraud[test], s), 3))

y = df.fraud[test].values; amt = df.amount[test].values; review = 5.0
print("flag nothing:", round(amt[y == 1].sum()), " flag everything:", round(review * len(y)))
best = None
for th in [0.01, 0.02, 0.03, 0.05, 0.1, 0.2, 0.3, 0.5]:
    f = scores["with graph"] >= th
    cost = review * f.sum() + amt[(~f) & (y == 1)].sum()
    print("threshold", th, "flagged", int(f.sum()), "caught", int((f & (y == 1)).sum()), "of", int(y.sum()), "cost", round(cost))

amount+night AUC 0.771 AP 0.053
with graph AUC 0.867 AP 0.542
flag nothing: 47034  flag everything: 100000
threshold 0.01 flagged 2756 caught 158 of 212 cost 19268
threshold 0.02 flagged 1443 caught 143 of 212 cost 15864
threshold 0.03 flagged 956 caught 131 of 212 cost 16762
threshold 0.05 flagged 331 caught 114 of 212 cost 20109
threshold 0.1 flagged 144 caught 107 of 212 cost 21749
threshold 0.2 flagged 124 caught 102 of 212 cost 23265
threshold 0.3 flagged 119 caught 100 of 212 cost 23772
threshold 0.5 flagged 109 caught 97 of 212 cost 24374


## 11.10 Scorecards: Reject Inference

A credit model is trained on applicants you *approved*, because only they have a repayment outcome. But it will score *everyone* who applies. The rejected group is missing from training: this is **selection bias**, and fixing it is called **reject inference**.

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

rng = np.random.default_rng(3)
n = 60000
inc = rng.normal(0, 1, n); util = rng.normal(0, 1, n); hid = rng.normal(0, 1, n)
z = -2.0 - 0.9 * inc + 0.6 * util + 1.0 * hid
d = (rng.random(n) < 1 / (1 + np.exp(-z))).astype(int)
X = np.c_[inc, util]
tr = np.arange(n) < 40000; te = ~tr

def study(label, score):
    appr = score > 0
    mA = LogisticRegression(C=1e6).fit(X[tr & appr], d[tr & appr])
    mAll = LogisticRegression(C=1e6).fit(X[tr], d[tr])
    print(label, "approved", appr.mean().round(2), "bad rate approved", d[appr].mean().round(3), "rejected", d[~appr].mean().round(3))
    for nm, m in (("approved only", mA), ("all applicants", mAll)):
        print("  ", nm, "coefs", m.coef_[0].round(2), "AUC", round(roc_auc_score(d[te], m.predict_proba(X[te])[:, 1]), 3),
              "mean predicted PD", m.predict_proba(X[te])[:, 1].mean().round(3), "actual", d[te].mean().round(3))

study("Policy A (recorded only):", 0.8 * inc - 0.5 * util + rng.normal(0, 0.3, n))
study("Policy B (uses hidden view):", 0.5 * inc - 0.5 * util - 1.5 * hid + rng.normal(0, 0.3, n))

Policy A (recorded only): approved 0.5 bad rate approved 0.091 rejected 0.281
   approved only coefs [-0.8   0.58] AUC 0.738 mean predicted PD 0.192 actual 0.187
   all applicants coefs [-0.79  0.52] AUC 0.738 mean predicted PD 0.185 actual 0.187
Policy B (uses hidden view): approved 0.5 bad rate approved 0.062 rejected 0.308
   approved only coefs [-0.69  0.34] AUC 0.736 mean predicted PD 0.077 actual 0.187
   all applicants coefs [-0.79  0.52] AUC 0.738 mean predicted PD 0.185 actual 0.187


## Chapter recap

## Practice set

**1.** (Easy) Bin A has 40% of goods and 10% of bads. Find its WoE. **Answer:** ln(0.40/0.10) = ln 4 = 1.386. Very safe bin.

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/ds/ch11.html